<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

## Save DE Pacific Coastlines for a bounding box

The following operation saves annual shorelines and rates of change for a bounding box to a GeoPackage. This operation only reads the data for the bounding box, not the full data file.

Open it in QGIS and the official styles are applied automatically. The map below shows the same data from the Digital Earth Pacific tile server.

In [ ]:
import geopandas as gpd
from shapely.geometry import box


def load_coastlines(bbox, layer):
    """Load one layer of DE Pacific Coastlines inside a (min_lon, min_lat, max_lon, max_lat) box."""
    # URL to source data
    source = (
        "/vsicurl/https://dep-public-staging.s3.us-west-2.amazonaws.com/dep_ls_coastlines/"
        "processed/0-7-0-55/dep_ls_coastlines_0-7-0-55.gpkg"
    )
    # Styles are stored with placeholder geometry, so a box would filter them all out
    if layer == "layer_styles":
        return gpd.read_file(source, layer=layer)
    # A box with a CRS is reprojected to match the data, and only matching features are downloaded
    area = gpd.GeoSeries([box(*bbox)], crs=4326)
    return gpd.read_file(source, layer=layer, bbox=area)


# Funafuti, Tuvalu
bbox = (179.05, -8.65, 179.25, -8.45)
output_path = "dep_coastlines_funafuti.gpkg"

# Load our layers of interest and write them all to the same file.
load_coastlines(bbox, "shorelines_annual").to_file(output_path, layer="shorelines_annual")
load_coastlines(bbox, "rates_of_change").to_file(output_path, layer="rates_of_change")

# Write the style too, so it will appear in QGIS
load_coastlines(bbox, "layer_styles").to_file(output_path, layer="layer_styles")

## Load pre-rendered data

We don't have to work with the raw data at all. Data is served in multiple formats as
listed below. This only downloads pre-rendered data for the areas we are looking at.

In [ ]:
import folium

min_lon, min_lat, max_lon, max_lat = bbox
coastlines_map = folium.Map(tiles="Esri.WorldImagery")
folium.TileLayer(
    tiles="https://tileserver.prod.digitalearthpacific.io/styles/coastlines/{z}/{x}/{y}.png",
    attr="Digital Earth Pacific",
    overlay=True,
).add_to(coastlines_map)
coastlines_map.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
coastlines_map

The table below shows different versions of pre-rendered data that are available. These can be used in GIS software or
in a web map

Replace `coastlines` with `coastlines_annual` (shorelines only) or `coastlines_rates_of_change` (rate points only) for a single layer.

| What | URL |
|---|---|
| Styled image tiles (WMTS) | `https://tileserver.prod.digitalearthpacific.io/styles/coastlines/wmts.xml` |
| Styled image tiles (XYZ) | `https://tileserver.prod.digitalearthpacific.io/styles/coastlines/{z}/{x}/{y}.png` |
| Vector tiles (max zoom 13) | `https://tileserver.prod.digitalearthpacific.io/data/coastlines/{z}/{x}/{y}.pbf` |
| Vector tile style (MapLibre) | `https://tileserver.prod.digitalearthpacific.io/styles/coastlines/style.json` |
| Static image for a box | `https://tileserver.prod.digitalearthpacific.io/styles/coastlines/static/{min_lon},{min_lat},{max_lon},{max_lat}/{width}x{height}.png` |